# ДЗ 1. Оптимальная стратегия перемещения таксиста по городу Новинск

In [ ]:
!pip install gym==0.22.0

## Задание 1. Описание марковского процесса

### Среда

Таксист передвигается по сетке 5×5 ячеек. На сетке расположены 4 фиксированных места: **R** (0,0), **G** (0,4), **Y** (4,0), **B** (4,3). Они же выступают и как места посадки пассажиров (A, B, C, D), и как пункты назначения (a, b, c, d).

### Пространство состояний

$S = (\text{taxi\_row},\ \text{taxi\_col},\ \text{passenger\_location},\ \text{destination})$

- `taxi_row` ∈ {0, 1, 2, 3, 4}
- `taxi_col` ∈ {0, 1, 2, 3, 4}
- `passenger_location` ∈ {0 (R), 1 (G), 2 (Y), 3 (B), 4 (в такси)}
- `destination` ∈ {0 (R), 1 (G), 2 (Y), 3 (B)}

Итого: 5 × 5 × 5 × 4 = **500 состояний**

### Пространство действий

| Действие | Описание |
|----------|----------|
| 0 — South | Вниз |
| 1 — North | Вверх |
| 2 — East | Вправо |
| 3 — West | Влево |
| 4 — Pickup | Подобрать пассажира |
| 5 — Dropoff | Высадить пассажира |

### Функция переходов

Переходы **детерминированные**: $P(s' | s, a) = 1.0$. Таксист автоматически не подбирает пассажира — для этого нужно выбрать действие Pickup. Стены блокируют движение (такси остаётся на месте).

### Функция вознаграждения

| Событие | Вознаграждение |
|---------|---------------|
| Каждый шаг | −1 |
| Успешная высадка | +20 |
| Неверный Pickup / Dropoff | −10 |

### Начальное состояние

В начале каждого эпизода случайным образом выбирается позиция такси, место пассажира и пункт назначения.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import gym

env = gym.make("Taxi-v3")
env.reset()
env.render()
print(f"Состояний: {env.observation_space.n}, Действий: {env.action_space.n}")

## Задание 2. Value Iteration (Вариант 1: mdp_taxi_v1)

In [ ]:
class TaxiMDPWrapper:
    def __init__(self, env_name='Taxi-v3'):
        self.env = gym.make(env_name)
        self.states = list(range(self.env.observation_space.n))
        self.actions = list(range(self.env.action_space.n))
        self.transitions = self.env.P
        self._initial_state = self.env.reset()

    def get_all_states(self):
        return [state for state in self.states]

    def get_possible_actions(self, state):
        return list(range(len(self.actions)))

    def get_next_states(self, state, action):
        p, state_next, _, _ = self.transitions[state][action][0]
        return {state_next: 1.0}

    def get_reward(self, state, action, state_next):
        _, _, reward, _ = self.transitions[state][action][0]
        return reward

    def get_transition_prob(self, state, action, state_next):
        p, next_state, _, _ = self.transitions[state][action][0]
        return p

    def is_terminal(self, state):
        done = self.transitions[state][0][0][3]
        return done

In [ ]:
def get_action_value(mdp, state_values, state, action, gamma):
    Q = sum([next_prob * (mdp.get_reward(state, action, next_state) + gamma * state_values[next_state])
              for next_state, next_prob in mdp.get_next_states(state, action).items()])
    return Q

In [ ]:
def get_new_state_value(mdp, state_values, state, gamma):
    if mdp.is_terminal(state):
        return 0
    V = max([get_action_value(mdp, state_values, state, a, gamma) for a in mdp.get_possible_actions(state)])
    return V

In [ ]:
def value_iteration(mdp, state_values=None,
          gamma=0.9, num_iter=1000, min_difference=1e-5):
    state_values = state_values or \
      {s : 0 for s in mdp.get_all_states()}
    history = []

    for i in range(num_iter):
        new_state_values = {s: get_new_state_value(mdp, state_values, s, gamma) for s in mdp.get_all_states()}

        assert isinstance(new_state_values, dict)

        diff = max(abs(new_state_values[s] - state_values[s]) for s in mdp.get_all_states())

        print("iter %4i | diff: %6.5f | V(start): %.3f " %
          (i, diff, new_state_values[mdp._initial_state]))

        state_values = new_state_values
        history.append(state_values[mdp._initial_state])

        if diff < min_difference:
            print("Алгоритм сходится!")
            break

    return state_values, history

In [ ]:
def get_optimal_action(mdp, state_values, state,
                       gamma=0.9):
    if mdp.is_terminal(state): return None

    actions = mdp.get_possible_actions(state)
    i = np.argmax([get_action_value(mdp, state_values, state, a, gamma) for a in actions])

    return actions[i]

In [ ]:
mdp = TaxiMDPWrapper()
state_values, history = value_iteration(mdp, gamma=0.99, num_iter=200, min_difference=1e-5)

## Задание 3. Демонстрация работы обученной стратегии

### Маршрут агента

In [ ]:
ACTION_NAMES = ['South', 'North', 'East', 'West', 'Pickup', 'Dropoff']
GAMMA = 0.99

def run_episode(mdp, state_values, gamma=GAMMA, render=True):
    env = gym.make("Taxi-v3")
    s = env.reset()
    total_reward = 0
    trajectory = [s]

    if render:
        env.render()

    for t in range(200):
        a = get_optimal_action(mdp, state_values, s, gamma)
        if a is None:
            break
        if render:
            print(ACTION_NAMES[a], end='\n\n')

        s, r, done, _ = env.step(a)
        total_reward += r
        trajectory.append(s)

        if render:
            env.render()

        if done:
            break

    if render:
        print(f"Total reward: {total_reward}")

    env.close()
    return trajectory, total_reward

trajectory, total_reward = run_episode(mdp, state_values)

In [ ]:
def decode_state(i):
    out = []
    out.append(i % 4)
    i = i // 4
    out.append(i % 5)
    i = i // 5
    out.append(i % 5)
    i = i // 5
    out.append(i)
    return list(reversed(out))

def draw_taxi_route(trajectory):
    positions = []
    for s in trajectory:
        taxi_row, taxi_col, _, _ = decode_state(s)
        positions.append((taxi_row, taxi_col))

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    rows = [p[0] for p in positions]
    cols = [p[1] for p in positions]

    ax.plot(cols, rows, 'b-o', markersize=6, linewidth=2, alpha=0.6)
    ax.plot(cols[0], rows[0], 'gs', markersize=15, label='Start')
    ax.plot(cols[-1], rows[-1], 'r*', markersize=20, label='End')

    locs = [(0, 0, 'R'), (0, 4, 'G'), (4, 0, 'Y'), (4, 3, 'B')]
    for r, c, name in locs:
        ax.annotate(name, (c, r), fontsize=14, fontweight='bold', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))

    ax.set_xlim(-0.5, 4.5)
    ax.set_ylim(4.5, -0.5)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.grid(True)
    ax.set_title("Маршрут таксиста")
    ax.legend()
    plt.show()

draw_taxi_route(trajectory)

### Демонстрация на различных вариантах поля

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, ax in enumerate(axes):
    traj, rew = run_episode(mdp, state_values, render=False)
    positions = [decode_state(s) for s in traj]
    rows = [p[0] for p in positions]
    cols = [p[1] for p in positions]

    ax.plot(cols, rows, 'b-o', markersize=5, linewidth=2, alpha=0.6)
    ax.plot(cols[0], rows[0], 'gs', markersize=12, label='Start')
    ax.plot(cols[-1], rows[-1], 'r*', markersize=18, label='End')

    locs = [(0, 0, 'R'), (0, 4, 'G'), (4, 0, 'Y'), (4, 3, 'B')]
    for r, c, name in locs:
        ax.annotate(name, (c, r), fontsize=12, fontweight='bold', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))

    ax.set_xlim(-0.5, 4.5)
    ax.set_ylim(4.5, -0.5)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.grid(True)
    ax.set_title(f"Эпизод {idx + 1}, reward = {rew}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### Сходимость Value Function

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history, 'b-', linewidth=2)
plt.xlabel('Итерация')
plt.ylabel('V(s₀)')
plt.title('Сходимость Value Function по итерациям')
plt.grid(True)
plt.show()

### Среднее вознаграждение агента

In [ ]:
rewards = []
for _ in range(1000):
    _, r = run_episode(mdp, state_values, render=False)
    rewards.append(r)

print(f"Average reward over 1000 episodes: {np.mean(rewards):.2f}")

plt.figure(figsize=(10, 4))
plt.hist(rewards, bins=30, edgecolor='black')
plt.axvline(np.mean(rewards), color='r', linestyle='--', label=f'Mean = {np.mean(rewards):.2f}')
plt.xlabel('Reward')
plt.ylabel('Count')
plt.title('Распределение вознаграждений обученной стратегии')
plt.legend()
plt.grid(True)
plt.show()